# 双通道 FRET：人工复筛

自动判据（`smTest_dual_batch` 那一套）只能看数字，看不出"孔里有第二个分子在闪""背景环被
邻居污染""漂白点落在一次闪光的尾巴上"这类结构问题。粗筛已经把每视野压到 **10 条上下**，
摊开看一眼比继续调阈值划算。

- **输入**：`<sif 主名>_fret/` 里的 `fret.csv`（E 与判据量）＋ `good_traces.csv`（trace 本体）
- **输出**：同一目录下的 `*_manual.*`，**机器筛的产物一个字节都不动**
  | 文件 | 内容 |
  |---|---|
  | `fret_manual.csv` | 人工筛后的逐分子 E（列与 `fret.csv` 相同） |
  | `E_distribution_manual.png` | 人工筛后的 E 分布 + 两个交叉检查 |
  | `good_traces_manual.png` | 留下的 trace 总览 |
  | `good_traces_manual.csv` | 留下的 trace（可再读回） |
  | `rejected_traces_manual.png` | **被删掉的** trace 总览（留证据） |

γ/α/β **沿用 `fret.csv` 里的值**，这一步不重估 —— 否则人工筛完的 E 与机器筛的不可比。


In [ ]:
# `import smlm` 需要 Py_Smile 在 sys.path 上; notebook 放在子目录里时自己找一下
import pathlib
import sys

for _q in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_q / 'smlm' / '__init__.py').is_file():
        sys.path.insert(0, str(_q))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import smlm.fret as fr
from smlm.datatypes import Trace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ======================= 输入 =======================
BASE_DIR = pathlib.Path(r"G:\PacBio\近观DNA_FRET\20261123")
FIELDS = []            # 空 = BASE_DIR 下所有 *_fret 目录 (按名字排序)
SUFFIX = '_manual'     # 输出文件名后缀; 想让路给另一轮就改掉

# ======================= 人工剔除 =======================
# 键 = 视野名 (fov6 / sample32_..._fov6), 值 = 分子编号列表 (图上/表里的 #id)
# 下一格会把某个视野的全部 trace 画出来, 照着填即可。
REJECT = {
    # 'fov6': [65, 164],
}
SHOW = 'fov6'          # 下一格先看哪个视野

print('BASE_DIR :', BASE_DIR)
print('视野     :', ', '.join(
    [p.name[:-len("_fret")] for p in sorted(BASE_DIR.glob("*_fret"))]) or '(无)')


## 读回粗筛结果

列名与判据量都从 `fret.csv` 还原；trace 从 `good_traces.csv` 读，按 `emitter_id` 对齐。

In [ ]:
def field_dir(name):
    """<BASE_DIR>/<name>_fret, 不存在就报错 (不静默跳过)。"""
    d = BASE_DIR / f'{name}_fret'
    if not d.is_dir():
        raise FileNotFoundError(d)
    return d


def field_names():
    if FIELDS:
        return list(FIELDS)
    return sorted(p.name[:-len('_fret')] for p in BASE_DIR.glob('*_fret'))


def _trace_from(arr, k, emitter_id):
    idd = np.asarray(arr['I_donor'][k], float)
    ia = np.asarray(arr['I_acceptor'][k], float)
    return Trace(emitter_id=int(emitter_id), frames=np.arange(idd.size),
                 x=float(arr['x_px'][k, 0]), y=float(arr['y_px'][k, 0]),
                 intensity=idd, bg_flux=np.asarray(arr['bg_donor'][k], float),
                 raw=idd + np.asarray(arr['bg_donor'][k], float))


def load_field(name, verbose=True):
    """读回一个视野的粗筛结果。

    返回 ``(records, traces_d, traces_a, results, ex_time, cal)``:
    ``records`` 全部 ``keep=True`` (E 与判据量都从 csv 还原),
    ``traces_*`` 与 ``records`` 同序 —— **下标不等于 emitter_id**, 画图按 id 取。
    """
    d = field_dir(name)
    df = pd.read_csv(d / fr.FILES['fret_csv'], encoding='utf-8-sig')
    arr = fr.load_traces_csv(d / fr.FILES['traces_good'])
    ids = [int(v) for v in arr['emitter_id']]
    pos = {v: k for k, v in enumerate(ids)}

    traces_d, traces_a, recs, res = [], [], [], []
    for _, row in df.iterrows():
        i = int(row['idx'])
        if i not in pos:
            raise KeyError(f'{name}: traces csv 里没有 #{i}, 与 fret.csv 对不上')
        k = pos[i]
        traces_d.append(_trace_from(arr, k, i))
        traces_a.append(Trace(**{**dict(traces_d[-1]),
                                 'intensity': np.asarray(arr['I_acceptor'][k], float),
                                 'bg_flux': np.asarray(arr['bg_acceptor'][k], float),
                                 'raw': np.asarray(arr['I_acceptor'][k], float)
                                 + np.asarray(arr['bg_acceptor'][k], float)}))

        ev = fr.StepEvent(emitter_id=i, frame=int(row['frame_j']),
                          lag=int(row['lag']),
                          IA_pre=float(row['I_A_pre']),
                          IA_post=float(row['I_A_post']),
                          ID_pre=float(row['I_D_pre']),
                          ID_post=float(row['I_D_post']),
                          dIA=float(row['I_A_pre'] - row['I_A_post']),
                          dID=float(row['I_D_post'] - row['I_D_pre']),
                          dID_long=np.nan)
        rec = fr.PairRecord(emitter_id=i, keep=True, event=ev,
                            E=float(row['E']),
                            spread=float(row['spread']),
                            n_d_pre=int(row['n_d_pre']),
                            red_label=str(row['red_label']),
                            red_duty=float(row['red_duty']),
                            IA_max=float(row['I_A_pre']))
        res.append(fr.FretResult(
            emitter_id=i, frame=int(row['frame_j']), lag=int(row['lag']),
            n_pre=int(row['n_pre']), n_on=int(row['n_on']),
            IA_pre=float(row['I_A_pre']), ID_pre=float(row['I_D_pre']),
            IA_post=float(row['I_A_post']), ID_post=float(row['I_D_post']),
            S_A=float(row['S_A']), E=float(row['E']),
            E_don=float(row['E_don']), snr_A=float(row['snr_A']),
            t_bleach=float(row['t_bleach_s'])))
        recs.append(rec)

    ex = float(arr['t_s'][0, 1] - arr['t_s'][0, 0])
    cal = fr.FretCalibration(gamma=float(df['gamma'].iloc[0]),
                             alpha=float(df['alpha'].iloc[0]),
                             beta=float(df['beta'].iloc[0]))
    if verbose:
        print(f'{name}: {len(recs)} 条粗筛通过   γ={cal.gamma:.2f} '
              f'α={cal.alpha:.3f} β={cal.beta:+.1f}   {ex * 1000:.0f} ms/帧')
    return recs, traces_d, traces_a, res, ex, cal


## 逐个视野看

粗筛通过项一次全画出来（每视野 10 条左右，4 列会自动折行）。标题里是 `#id / E / I_A / I_D / lag`，下面的表把判据量也列出来。

**要删的编号填进上一格的 `REJECT`，重跑这一格确认，再跑下一格。**

In [ ]:
# 把 SHOW 这个视野的全部粗筛通过项画出来 —— 要删的把 #id 记下来, 填进 REJECT
recs, traces_d, traces_a, results, EX, CAL = load_field(SHOW)

fig = fr.viz.plot_pair_gallery(recs, traces_a, traces_d, which='kept',
                               n=len(recs), ncols=4, ex_time=EX)
fig.suptitle(f'{SHOW}: {len(recs)} 条粗筛通过'
             f'  (红=受体, 绿=供体; 阴影 = 算 E 的 pre/post 窗)',
             fontsize=11, y=1.0)
plt.show()

decide = pd.DataFrame([{
    'id': r.emitter_id,
    't_bleach_s': round(results[k].t_bleach, 1),
    'E': round(r.E, 3),
    'E_don': round(results[k].E_don, 3),
    'I_A_pre': round(results[k].IA_pre, 0),
    'I_D_pre': round(results[k].ID_pre, 0),
    'I_D_post': round(results[k].ID_post, 0),
    'snr_A': round(results[k].snr_A, 1),
    'red': r.red_label,
    'duty': round(r.red_duty, 2),
    'spread': round(r.spread, 2),
    'n_d^pre': r.n_d_pre,
} for k, r in enumerate(recs)]).sort_values('E')
print(decide.to_string(index=False))
print()
print(f'当前 REJECT[{SHOW!r}] = {REJECT.get(SHOW, [])}')


## 应用到全部视野并落盘

`include_rejected=True` 会额外存一张被删掉的 trace，以后想复核"当初为什么删"有据可查。

In [ ]:
# 按 REJECT 重新落盘: 每个视野单独一套 *_manual.*
rows, all_res, all_ids = [], [], []

for name in field_names():
    recs, traces_d, traces_a, results, EX, CAL = load_field(name, verbose=False)
    drop = {int(v) for v in REJECT.get(name, [])}
    unknown = drop - {r.emitter_id for r in recs}
    if unknown:
        print(f'  [!] {name}: REJECT 里的 {sorted(unknown)} 不在这个视野里, 忽略')

    for r in recs:
        if r.emitter_id in drop:
            r.keep = False
            r.reasons = ['manual']

    keep = [r for r in recs if r.keep]
    kept_res = [r for r in results if r.emitter_id not in drop]
    E_before = np.array([r.E for r in results], float)
    E_after = np.array([r.E for r in kept_res], float)

    fr.save_bundle(field_dir(name), records=recs, results=kept_res,
                   traces_d=traces_d, traces_a=traces_a, cal=CAL, ex_time=EX,
                   title=name, include_all=False, include_rejected=True,
                   suffix=SUFFIX, ncols=4, verbose=False)

    all_res += kept_res
    all_ids += [(name, r.emitter_id) for r in kept_res]
    rows.append(dict(
        field=name, n_before=len(recs), n_drop=len(drop), n_after=len(keep),
        n_E=len(kept_res),
        E_med_before=float(np.median(E_before)),
        E_med_after=float(np.median(E_after)) if E_after.size else np.nan,
        E_p16=float(np.percentile(E_after, 16)) if E_after.size else np.nan,
        E_p84=float(np.percentile(E_after, 84)) if E_after.size else np.nan,
    ))
    print(f'{name:<10} {len(recs):>3} -> {len(keep):>3} 条 '
          f'(删 {len(drop)})   E 中位 {rows[-1]["E_med_before"]:.3f} -> '
          f'{rows[-1]["E_med_after"]:.3f}   -> {field_dir(name).name}/')

summary = pd.DataFrame(rows)
print()
print(summary.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print(f'\n合计留下 {len(all_res)} 条 E')


## 人工筛后的池化分布

In [ ]:
# 人工筛后的池化 E 分布 (与 smTest_dual_batch 最后一格同一画法)
if not all_res:
    print('没有留下任何 E')
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fr.viz.plot_efficiency(all_res, ax=axes[0])
    fr.viz.plot_efficiency_vs_brightness(all_res, ax=axes[1], key='IA_pre')
    fig.suptitle(f'{len(rows)} fields   smFRET (manual{SUFFIX})   '
                 f'n={len(all_res)}', fontsize=11)
    fig.tight_layout()
    out = BASE_DIR / 'FRET_summary' / f'E_all_fields{SUFFIX}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'池化图 -> {out}')

    E = np.array([r.E for r in all_res], float)
    print(f'E 中位 {np.median(E):.3f}   p16~p84 '
          f'{np.percentile(E, 16):.3f}~{np.percentile(E, 84):.3f}   '
          f'E<0: {int((E < 0).sum())}   E>1: {int((E > 1).sum())}')
